# 进阶实践项目参考答案 00：基础编程与人工智能：稳健性与概率评价

在手写数字分类之后，继续检查模型面对噪声、类别不平衡和概率解释时是否可靠。

Kaggle 中先复制到自己的账户，再按任务顺序完成。题目只保留关键填写位置，数据读取、绘图和保存框架已经给出。

## 任务
1. 完成分层划分与标准化
2. 训练逻辑回归和随机森林
3. 计算宏平均 F1 与混淆矩阵
4. 绘制可靠性曲线
5. 比较加噪前后的性能

In [1]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
SEED=42; OUT=Path('advanced00_results'); OUT.mkdir(exist_ok=True)
X,y=load_digits(return_X_y=True)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
models={'logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),'forest':RandomForestClassifier(n_estimators=220,min_samples_leaf=2,random_state=SEED,n_jobs=-1)}
metrics={}
for name,model in models.items():
    model.fit(X_train,y_train); pred=model.predict(X_test); prob=model.predict_proba(X_test)
    metrics[name]={'accuracy':float(accuracy_score(y_test,pred)),'macro_f1':float(f1_score(y_test,pred,average='macro'))}
best=models[max(metrics,key=lambda k:metrics[k]['macro_f1'])]
pred=best.predict(X_test); prob=best.predict_proba(X_test); confidence=prob.max(1); correct=(pred==y_test).astype(int)
frac,mean=calibration_curve(correct,confidence,n_bins=8,strategy='uniform')
rng=np.random.default_rng(SEED); noisy=np.clip(X_test+rng.normal(0,2.2,X_test.shape),0,16)
noisy_pred=best.predict(noisy); metrics['noise']={'accuracy':float(accuracy_score(y_test,noisy_pred)),'macro_f1':float(f1_score(y_test,noisy_pred,average='macro'))}
fig,ax=plt.subplots(1,2,figsize=(10,4)); ConfusionMatrixDisplay.from_predictions(y_test,pred,ax=ax[0],colorbar=False); ax[0].set_title('Test confusion matrix'); ax[1].plot([0,1],[0,1],'--'); ax[1].plot(mean,frac,'o-'); ax[1].set(xlabel='Mean confidence',ylabel='Observed accuracy',title='Reliability curve'); fig.tight_layout(); fig.savefig(OUT/'advanced00_summary.png',dpi=150); plt.close(fig)
(OUT/'advanced00_result.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
print(metrics)


{'logistic': {'accuracy': 0.9777777777777777, 'macro_f1': 0.9776411985388958}, 'forest': {'accuracy': 0.9711111111111111, 'macro_f1': 0.9707197698826022}, 'noise': {'accuracy': 0.7466666666666667, 'macro_f1': 0.75680943541131}}
